# Unsupervised cell clustering

This notebook looks for hidden macrophage sub-populations without giving the algorithm any treatment labels. Every cell is represented only by **CD206 brightness** and **cell area**, then grouped with a simple NumPy implementation of **K-means clustering**.

The clustering is fit **per donor**, but not with fully independent random starts anymore. Instead, the notebook fits a **reference donor** first in a shared standardized feature space, then uses those final centroids to initialize every other donor. That keeps cluster identities much more stable across donors during the simulation itself.

The workflow has two parts:
1. an interactive simulation that shows donor-specific centroids moving while K-means converges
2. a reproducible post hoc analysis that reveals which treatments are enriched in each reference-aligned cluster


In [ ]:
import os
import sys
from pathlib import Path

PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if not (PROJECT_ROOT / "macrophage_analysis").exists():
    raise RuntimeError("Run this notebook from the repository root or notebooks/ directory.")
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "3"

import time

import ipywidgets as widgets
from IPython.display import clear_output, display

import macrophage_analysis as ma
from macrophage_analysis.analysis.clustering import (
    build_aligned_cluster_composition_table,
    build_donor_feature_data,
    build_split_kmeans_state_sequence,
    ordered_present_values,
    run_split_kmeans,
)
from macrophage_analysis.notebook import build_marker_dropdown
from macrophage_analysis.plotting.clustering import (
    plot_aligned_cluster_composition,
    plot_kmeans_state_grid,
)
from macrophage_analysis.analysis.morphology_tables import build_morphology_table

cluster_donors = list(ma.DEFAULT_DONORS)
cluster_conditions = list(ma.DEFAULT_CONDITIONS)
cluster_antibody_dropdown = build_marker_dropdown()
cluster_non_reference_mode = "independent"
cluster_max_alignment_distance = 2.0
cluster_n_tiles = (8, 8)

display(cluster_antibody_dropdown)
cluster_donors, cluster_conditions, cluster_antibody_dropdown.value


Run this once. The clustering notebook only extracts **CD206** so it stays focused on the two features used downstream: intensity and area.


In [ ]:
cluster_antibody = str(cluster_antibody_dropdown.value)

cluster_results = ma.extract_single_cell_fluorescence(
    donors=cluster_donors,
    conditions=cluster_conditions,
    antibody_order=[cluster_antibody],
    n_tiles=cluster_n_tiles,
)


Build the per-cell table that K-means will use. At this stage the treatment labels are still present in the DataFrame for later interpretation, but the algorithm itself will only see the `intensity` and `area` columns and will fit each donor separately.


In [ ]:
cluster_cells = build_morphology_table(
    cluster_results,
    antibody=cluster_antibody,
)

display(
    cluster_cells.groupby(["donor_label", "condition_label"], observed=True)
    .size()
    .rename("cell_count")
    .reset_index()
)


## Interactive donor-specific K-means simulation

Choose the number of clusters `K`, choose a **reference donor**, then click **Run Algorithm**. The reference donor is fit first. Every other donor starts from those final reference centroids in the same globally standardized CD206 / area space, so `C1`, `C2`, `C3`, ... stay much more consistent across donors during the run.

The plots are shown in the original CD206 / area space even though clustering happens in the standardized space.


In [ ]:
cluster_donor_feature_data = build_donor_feature_data(cluster_cells)
cluster_donor_label_order = list(cluster_donor_feature_data)
cluster_max_k = min(8, min(len(payload["feature_frame"]) for payload in cluster_donor_feature_data.values()))
k_slider = widgets.IntSlider(value=min(3, cluster_max_k), min=2, max=cluster_max_k, step=1, description="Clusters (K)")
seed_slider = widgets.IntSlider(value=7, min=0, max=999, step=1, description="Random seed")
iteration_slider = widgets.IntSlider(value=10, min=2, max=25, step=1, description="Max iterations")
delay_slider = widgets.FloatSlider(value=0.5, min=0.1, max=1.5, step=0.1, description="Delay (s)")
simulation_reference_donor_dropdown = widgets.Dropdown(
    options=cluster_donor_label_order,
    value=cluster_donor_label_order[0],
    description="Reference donor",
)
run_button = widgets.Button(description="Run Algorithm", button_style="primary")
simulation_output = widgets.Output()


def run_simulation(_button):
    run_button.disabled = True
    try:
        reference_donor_label = str(simulation_reference_donor_dropdown.value)
        donor_feature_data, donor_runs, clustered_cells, centroid_table = run_split_kmeans(
            cluster_cells,
            k=int(k_slider.value),
            random_seed=int(seed_slider.value),
            max_iterations=int(iteration_slider.value),
            reference_donor_label=reference_donor_label,
            non_reference_mode=cluster_non_reference_mode,
            max_alignment_distance=cluster_max_alignment_distance,
        )
        state_sequence = build_split_kmeans_state_sequence(donor_runs)
        for state_by_donor in state_sequence:
            with simulation_output:
                clear_output(wait=True)
                count_lines = plot_kmeans_state_grid(
                    donor_feature_data,
                    state_by_donor,
                    antibody=cluster_antibody,
                    k=int(k_slider.value),
                )
                for line in count_lines:
                    print(line)
            time.sleep(float(delay_slider.value))
    finally:
        run_button.disabled = False


run_button.on_click(run_simulation)
display(
    widgets.VBox(
        [
            k_slider,
            simulation_reference_donor_dropdown,
            seed_slider,
            iteration_slider,
            delay_slider,
            run_button,
            simulation_output,
        ]
    )
)


## Reveal the labels after donor-specific clustering

Once you have an idea of a useful `K` from the simulation, set the values below and run the next cells. This reruns K-means deterministically in the same way: the selected **reference donor** is fit first, and every other donor starts from those final reference centroids in globally standardized CD206 / area space.

The matching step is still kept afterward as a safety check, but in the ideal case the warm start already keeps the raw cluster numbers aligned.

Use the dropdown to choose which donor acts as the reference for the cross-donor alignment.


In [ ]:
final_cluster_k = 3
final_cluster_seed = 548
final_cluster_iterations = 10

reference_donor_options = ordered_present_values(cluster_cells["donor_label"])
reference_donor_dropdown = widgets.Dropdown(
    options=reference_donor_options,
    value=reference_donor_options[0],
    description="Reference donor",
)
display(reference_donor_dropdown)


In [ ]:
reference_donor_label = str(reference_donor_dropdown.value)

(
    final_donor_feature_data,
    final_donor_runs,
    clustered_cells,
    cluster_centroid_table,
) = run_split_kmeans(
    cluster_cells,
    k=final_cluster_k,
    random_seed=final_cluster_seed,
    max_iterations=final_cluster_iterations,
    reference_donor_label=reference_donor_label,
    non_reference_mode=cluster_non_reference_mode,
    max_alignment_distance=cluster_max_alignment_distance,
)

display(
    cluster_centroid_table[[
        "reference_donor_label",
        "donor_label",
        "cluster",
        "aligned_cluster",
        "alignment_distance",
        "cd206_intensity_centroid",
        "area_centroid",
    ]].round({"alignment_distance": 3, "cd206_intensity_centroid": 2, "area_centroid": 1})
)


In [ ]:
cluster_composition = build_aligned_cluster_composition_table(clustered_cells)

display(cluster_composition)
plot_aligned_cluster_composition(
    cluster_composition,
    reference_donor_label=reference_donor_label,
    cluster_count=final_cluster_k,
)
